In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Introducción al servicio de transpilación con IA de Qiskit

import TutorialFeedback from '@site/src/components/TutorialFeedback';





*Uso estimado de QPU: Ninguno (NOTA: Este tutorial no ejecuta trabajos porque se centra en la transpilación)*

## Antecedentes
El **servicio de transpilación con IA de Qiskit (QTS)** introduce optimizaciones basadas en aprendizaje automático tanto en los pases de enrutamiento como en los de síntesis. Estos modos de IA han sido diseñados para abordar las limitaciones de la transpilación tradicional, particularmente para circuitos a gran escala y topologías de hardware complejas.

A partir de **julio de 2025**, el **Servicio de Transpilación** ha sido migrado a la nueva plataforma IBM Quantum&reg; y ya no está disponible. Para las últimas actualizaciones sobre el estado del Servicio de Transpilación, consulta la [documentación del servicio de transpilación](/guides/qiskit-transpiler-service). Aún puede utilizar el transpilador con IA localmente, de manera similar a la transpilación estándar de Qiskit. Simplemente reemplace `generate_preset_pass_manager()` con `generate_ai_pass_manager()`. Esta función construye un gestor de pases que integra los pases de enrutamiento y síntesis con IA directamente en su flujo de trabajo de transpilación local.

### Características clave de los pases de IA
- Pases de enrutamiento: El enrutamiento con IA puede ajustar dinámicamente las rutas de los qubits según el circuito y el backend específicos, reduciendo la necesidad de compuertas SWAP excesivas.
    - `AIRouting`: Selección de diseño y enrutamiento de circuitos

- Pases de síntesis: Las técnicas de IA optimizan la descomposición de compuertas multi-qubit, minimizando el número de compuertas de dos qubits, que suelen ser más propensas a errores.
    - `AICliffordSynthesis`: Síntesis de compuertas Clifford
    - `AILinearFunctionSynthesis`: Síntesis de circuitos de funciones lineales
    - `AIPermutationSynthesis`: Síntesis de circuitos de permutación
    - `AIPauliNetworkSynthesis`: Síntesis de circuitos de redes de Pauli (solo disponible en el Servicio de Transpilación de Qiskit, no en el entorno local)

- Comparación con la transpilación tradicional: El transpilador estándar de Qiskit es una herramienta robusta que puede manejar un amplio espectro de circuitos cuánticos de manera efectiva. Sin embargo, cuando los circuitos crecen en escala o las configuraciones de hardware se vuelven más complejas, los pases de IA pueden ofrecer ganancias de optimización adicionales. Al utilizar modelos aprendidos para el enrutamiento y la síntesis, QTS refina aún más los diseños de circuitos y reduce la sobrecarga para tareas cuánticas desafiantes o a gran escala.

Este tutorial evalúa los modos de IA utilizando tanto pases de enrutamiento como de síntesis, comparando los resultados con la transpilación tradicional para resaltar dónde la IA ofrece ganancias de rendimiento.

Para más detalles sobre los pases de IA disponibles, consulta la [documentación de pases de IA](/guides/ai-transpiler-passes).

### ¿Por qué utilizar IA para la transpilación de circuitos cuánticos?
A medida que los circuitos cuánticos crecen en tamaño y complejidad, los métodos de transpilación tradicionales tienen dificultades para optimizar los diseños y reducir los conteos de compuertas de manera eficiente. Los circuitos más grandes, particularmente aquellos que involucran cientos de qubits, imponen desafíos significativos en el enrutamiento y la síntesis debido a las restricciones del dispositivo, la conectividad limitada y las tasas de error de los qubits.

Aquí es donde la transpilación con IA ofrece una solución potencial. Al aprovechar técnicas de aprendizaje automático, el transpilador con IA en Qiskit puede tomar decisiones más inteligentes sobre el enrutamiento de qubits y la síntesis de compuertas, lo que lleva a una mejor optimización de circuitos cuánticos a gran escala.

### Resultados breves de benchmarking
![Graph showing AI transpiler performance against Qiskit](../docs/images/tutorials/ai-transpiler-introduction/ai-transpiler-benchmarks.avif)

En las pruebas de benchmarking, el transpilador con IA produjo consistentemente circuitos menos profundos y de mayor calidad en comparación con el transpilador estándar de Qiskit. Para estas pruebas, utilizamos la estrategia predeterminada del gestor de pases de Qiskit, configurada con [`generate_preset_passmanager`]. Si bien esta estrategia predeterminada es a menudo efectiva, puede tener dificultades con circuitos más grandes o más complejos. En contraste, los pases con IA lograron una reducción promedio del 24% en los conteos de compuertas de dos qubits y una reducción del 36% en la profundidad del circuito para circuitos grandes (más de 100 qubits) al transpilar a la topología heavy-hex del hardware de IBM Quantum. Para más información sobre estos benchmarks, consulta este [blog](https://www.ibm.com/quantum/blog/qiskit-performance).

Este tutorial explora los beneficios clave de los pases de IA y cómo se comparan con los métodos tradicionales.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit.random import random_circuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit_ibm_transpiler import generate_ai_pass_manager
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
import matplotlib.pyplot as plt
from statistics import mean, stdev
import time
import logging

seed = 42


def transpile_with_metrics(pass_manager, circuit):
    """Transpile a circuit and return the result along with key metrics."""
    start = time.time()
    qc_out = pass_manager.run(circuit)
    elapsed = time.time() - start

    depth_2q = qc_out.depth(lambda x: x.operation.num_qubits == 2)
    gate_count = qc_out.size()

    return qc_out, {
        "depth_2q": depth_2q,
        "gate_count": gate_count,
        "time_s": round(elapsed, 3),
    }


def remap_to_contiguous(tqc):
    """Remap a transpiled circuit to use contiguous qubit indices.

    Transpiled circuits target specific physical qubits (e.g., qubit 45, 67)
    on a large backend. This remaps them to 0, 1, 2, ... so Aer only
    simulates the active qubits."""
    active = sorted(
        {tqc.find_bit(q).index for inst in tqc.data for q in inst.qubits}
    )
    qubit_map = {old: new for new, old in enumerate(active)}
    new_qc = QuantumCircuit(len(active))
    for inst in tqc.data:
        old_indices = [tqc.find_bit(q).index for q in inst.qubits]
        new_qc.append(inst.operation, [qubit_map[i] for i in old_indices])
    return new_qc


def build_mirror_circuit(tqc, simulate=True):
    """Build a mirror circuit: U followed by U-dagger, with measurements.

    The expected output is always |0...0>, so measuring the survival
    probability reveals how much noise each transpilation strategy adds.

    Args:
        tqc: A transpiled circuit.
        simulate: If True (default), remap to contiguous qubits so Aer
            only simulates the active qubits. If False, keep the full
            physical layout for hardware execution."""
    if simulate:
        tqc = remap_to_contiguous(tqc)
    mirror = tqc.compose(tqc.inverse())
    mirror.measure_all()
    return mirror


def print_summary(results):
    """Print a summary of each metric as mean +/- stdev across all circuits,
    along with the mean percentage improvement of AI over Default."""
    metrics = [
        ("Depth 2Q", "Depth 2Q (Default)", "Depth 2Q (AI)"),
        ("Gate Count", "Gate Count (Default)", "Gate Count (AI)"),
        ("Time (s)", "Time (Default)", "Time (AI)"),
    ]
    header = (
        f"{'Metric':<12}{'Default (mean +/- std)':>24}"
        f"{'AI (mean +/- std)':>22}{'AI % improvement':>22}"
    )
    print(header)
    print("-" * len(header))
    for label, col_def, col_ai in metrics:
        defaults = [r[col_def] for r in results]
        ais = [r[col_ai] for r in results]
        pct = [(d - a) / d * 100 for d, a in zip(defaults, ais)]
        default_str = f"{mean(defaults):.1f} +/- {stdev(defaults):.1f}"
        ai_str = f"{mean(ais):.1f} +/- {stdev(ais):.1f}"
        pct_str = f"{mean(pct):+.1f}% +/- {stdev(pct):.1f}%"
        print(f"{label:<12}{default_str:>24}{ai_str:>22}{pct_str:>22}")


def plot_metrics_and_pct(results, title_prefix):
    """Plot metric comparisons and percentage improvement of AI over Default."""
    qubits = [r["Qubits"] for r in results]
    metrics = [
        ("Depth 2Q (Default)", "Depth 2Q (AI)", "Two-Qubit Depth"),
        ("Gate Count (Default)", "Gate Count (AI)", "Gate Count"),
        ("Time (Default)", "Time (AI)", "Transpilation Time"),
    ]

    # Row 1: raw metric comparison
    fig, axs = plt.subplots(1, 3, figsize=(21, 5))
    fig.suptitle(
        f"{title_prefix}: Metric Comparison",
        fontsize=15,
        fontweight="bold",
        y=1.02,
    )
    for ax, (col_def, col_ai, label) in zip(axs, metrics):
        ax.plot(qubits, [r[col_def] for r in results], "o-", label="Default")
        ax.plot(qubits, [r[col_ai] for r in results], "s-", label="AI")
        ax.set_title(label)
        ax.set_xlabel("Number of Qubits")
        ax.set_ylabel(label)
        ax.legend()
    plt.tight_layout()
    plt.show()

    # Row 2: percentage improvement
    fig, axs = plt.subplots(1, 3, figsize=(21, 5))
    fig.suptitle(
        f"{title_prefix}: % Improvement of AI over Default",
        fontsize=15,
        fontweight="bold",
        y=1.02,
    )
    for ax, (col_def, col_ai, label) in zip(axs, metrics):
        pct = [(r[col_def] - r[col_ai]) / r[col_def] * 100 for r in results]
        ax.axhline(
            0, color="#1f77b4", linewidth=2, label="Default (baseline)"
        )
        ax.plot(qubits, pct, "s-", color="#ff7f0e", label="AI")
        ax.fill_between(qubits, 0, pct, alpha=0.15, color="#ff7f0e")
        ax.set_title(label)
        ax.set_xlabel("Number of Qubits")
        ax.set_ylabel("% Improvement")
        ax.legend()
    plt.tight_layout()
    plt.show()


# Suppress verbose AI-powered transpiler logs
logging.getLogger(
    "qiskit_ibm_transpiler.wrappers.ai_local_synthesis"
).setLevel(logging.WARNING)

## Requisitos

Antes de comenzar este tutorial, asegúrate de tener lo siguiente instalado:

* Qiskit SDK v1.0 o posterior, con soporte de [visualización](https://docs.quantum.ibm.com/api/qiskit/visualization)
* Qiskit Runtime (`pip install qiskit-ibm-runtime`) v0.22 o posterior
* Qiskit IBM&reg; Transpiler con modo local de IA (`pip install 'qiskit-ibm-transpiler[ai-local-mode]'`)

## Configuración

In [2]:
num_circuits_sim = 20
depth_sim = 4
qubit_range_sim = list(range(6, 26))

circuits_sim = [
    # We have only two qubit gates, as those test how well the transpiler can optimize the circuit.
    random_circuit(
        num_qubits=n,
        depth=depth_sim,
        max_operands=2,
        num_operand_distribution={2: 1},
        seed=seed + i,
    )
    for i, n in enumerate(qubit_range_sim)
]

print(
    f"Created {len(circuits_sim)} circuits with qubit counts: {qubit_range_sim}"
)
circuits_sim[0].draw(output="mpl", fold=-1)

Created 20 circuits with qubit counts: [6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25]


<Image src="../docs/images/tutorials/ai-transpiler-introduction/extracted-outputs/sim-step1-code-1.avif" alt="Output of the previous code cell" />

# Parte I. Patrones de Qiskit
Veamos ahora cómo utilizar el servicio de transpilación con IA con un circuito cuántico simple, usando patrones de Qiskit. La clave es crear un `PassManager` con `generate_ai_pass_manager()` en lugar del estándar `generate_preset_pass_manager()`.
## Paso 1: Mapear entradas clásicas a un problema cuántico
En esta sección, probaremos el transpilador con IA en el circuito `efficient_su2`, un ansatz eficiente en hardware ampliamente utilizado. Este circuito es particularmente relevante para algoritmos cuánticos variacionales (por ejemplo, VQE) y tareas de aprendizaje automático cuántico, lo que lo convierte en un caso de prueba ideal para evaluar el rendimiento de la transpilación.

El circuito `efficient_su2` consiste en capas alternadas de rotaciones de un solo qubit y compuertas de entrelazamiento como CNOTs. Estas capas permiten una exploración flexible del espacio de estados cuánticos mientras mantienen la profundidad de compuertas manejable. Al optimizar este circuito, buscamos reducir el conteo de compuertas, mejorar la fidelidad y minimizar el ruido. Esto lo convierte en un candidato sólido para probar la eficiencia del transpilador con IA.

In [ ]:
service = QiskitRuntimeService()
backend = service.least_busy(
    min_num_qubits=100, operational=True, simulator=False
)


pm_default_sim = generate_preset_pass_manager(
    optimization_level=3,
    backend=backend,
    seed_transpiler=seed,
)

In [4]:
results_sim = []

for i, qc in enumerate(circuits_sim):
    n = qubit_range_sim[i]

    qc_default, m_default = transpile_with_metrics(pm_default_sim, qc)

    # Create a fresh AI pass manager each iteration to avoid stale layout state
    pm_ai = generate_ai_pass_manager(
        optimization_level=1,
        ai_optimization_level=3,
        backend=backend,
    )
    qc_ai, m_ai = transpile_with_metrics(pm_ai, qc)

    results_sim.append(
        {
            "Qubits": n,
            "Depth 2Q (Default)": m_default["depth_2q"],
            "Depth 2Q (AI)": m_ai["depth_2q"],
            "Gate Count (Default)": m_default["gate_count"],
            "Gate Count (AI)": m_ai["gate_count"],
            "Time (Default)": m_default["time_s"],
            "Time (AI)": m_ai["time_s"],
        }
    )

print_summary(results_sim)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Metric        Default (mean +/- std)     AI (mean +/- std)      AI % improvement
--------------------------------------------------------------------------------
Depth 2Q               33.0 +/- 12.9          26.4 +/- 8.0      +15.8% +/- 17.6%
Gate Count           522.0 +/- 266.0       560.5 +/- 279.1        -9.0% +/- 9.0%
Time (s)                 0.0 +/- 0.0           0.2 +/- 0.1    -893.6% +/- 362.9%


The summary table shows the mean and standard deviation of each metric across all 20 circuits, along with the average percentage improvement of the AI-powered transpiler over the default. Positive values indicate the AI-powered transpiler produced better results; negative values indicate the default was better.

For this small-scale example, the AI-powered transpiler achieves roughly 16% lower two-qubit depth on average, but at the cost of roughly 9% higher gate count. This highlights a key trade-off when choosing between the two strategies: the AI-powered transpiler prioritizes depth reduction (fewer sequential layers of two-qubit gates), while the default transpiler (SABRE) prioritizes minimizing total gate count (fewer SWAP insertions). Depending on your application, one metric may matter more than the other.

In [5]:
plot_metrics_and_pct(results_sim, "Small-Scale Random Circuits")

<Image src="../docs/images/tutorials/ai-transpiler-introduction/extracted-outputs/sim-step2-plot-0.avif" alt="Output of the previous code cell" />

<Image src="../docs/images/tutorials/ai-transpiler-introduction/extracted-outputs/sim-step2-plot-1.avif" alt="Output of the previous code cell" />

### Crear gestores de pases con IA y tradicionales
Para evaluar la efectividad del transpilador con IA, realizaremos dos ejecuciones de transpilación. Primero, transpilaremos el circuito utilizando el transpilador con IA. Luego, ejecutaremos una comparación transpilando el mismo circuito sin el transpilador con IA, utilizando métodos tradicionales. Ambos procesos de transpilación utilizarán el mismo mapa de acoplamiento del backend elegido y el nivel de optimización establecido en 3 para una comparación justa.

Ambos métodos reflejan el enfoque estándar para crear instancias de `PassManager` para transpilar circuitos en Qiskit.

In [6]:
# Use the 10-qubit circuit (index where qubits == 10)
idx_10q = qubit_range_sim.index(10)

qc_10q = circuits_sim[idx_10q]
qc_default_10q, _ = transpile_with_metrics(pm_default_sim, qc_10q)

pm_ai = generate_ai_pass_manager(
    optimization_level=1,
    ai_optimization_level=3,
    backend=backend,
)
qc_ai_10q, _ = transpile_with_metrics(pm_ai, qc_10q)

tqc_methods = {
    "Default": qc_default_10q,
    "AI": qc_ai_10q,
}

print(
    f"Default: depth {qc_default_10q.depth()}, gates {qc_default_10q.size()}"
)
print(f"AI:      depth {qc_ai_10q.depth()}, gates {qc_ai_10q.size()}")

Default: depth 84, gates 280
AI:      depth 91, gates 343


In [7]:
# Build a simple depolarizing noise model
noise_model = NoiseModel()
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(0.001, 1),
    ["sx", "x", "rz"],  # ~0.1% per 1Q gate
)
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(0.01, 2),
    ["cx", "ecr"],  # ~1% per 2Q gate
)

aer_sim = AerSimulator(noise_model=noise_model)

shots = 10000
survival_probs = {}

for method, tqc in tqc_methods.items():
    mirror = build_mirror_circuit(tqc, simulate=True)

    sampler = SamplerV2(mode=aer_sim)
    job = sampler.run([mirror], shots=shots)
    counts = job.result()[0].data.meas.get_counts()

    all_zeros = "0" * mirror.num_qubits
    survival = counts.get(all_zeros, 0) / shots
    survival_probs[method] = survival
    print(
        f"{method:8s}  P(|00...0>) = {survival:.4f}  ({counts.get(all_zeros, 0)}/{shots})"
    )

Default   P(|00...0>) = 0.8460  (8460/10000)
AI        P(|00...0>) = 0.8121  (8121/10000)


We ran both mirror circuits through the Aer simulator with a simple depolarizing noise model. The survival probability, defined as the fraction of shots that return the all-zeros bitstring, quantifies how much noise each transpilation strategy introduces.

### Step 4: Post-process and return result in desired classical format

We extract the probability of measuring the all-zeros bitstring from both runs. A higher survival probability indicates better fidelity, meaning the transpilation introduced less noise. The plot below shows the complement, 1 - P(|0...0>), so that a lower bar indicates better fidelity and small differences in error are easier to see.

In [8]:
# Plot 1 - P(|0...0>), the probability of an erroneous (non-zero) outcome.
# A lower bar means the transpilation introduced less noise.
error_probs = {method: 1 - p for method, p in survival_probs.items()}

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(
    error_probs.keys(),
    error_probs.values(),
    color=["steelblue", "coral"],
)
ax.set_ylabel("1 - P(|0...0>)")
ax.set_title("Mirror Circuit Error (10-qubit, Aer Simulator)")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

<Image src="../docs/images/tutorials/ai-transpiler-introduction/extracted-outputs/sim-step4-code-0.avif" alt="Output of the previous code cell" />

En esta prueba, comparamos el rendimiento del transpilador con IA y el método de transpilación estándar en el circuito efficient_su2. El transpilador con IA logra una profundidad de circuito notablemente menor mientras mantiene un conteo de compuertas similar.

- **Profundidad del circuito:** El transpilador con IA produce un circuito con menor profundidad de dos qubits. Esto es esperado, ya que los pases de IA están entrenados para optimizar la profundidad aprendiendo patrones de interacción de qubits y explotando la conectividad del hardware de manera más efectiva que las heurísticas basadas en reglas.

- **Conteo de compuertas:** El conteo total de compuertas permanece similar entre los dos métodos. Esto se alinea con las expectativas ya que la transpilación estándar basada en SABRE minimiza explícitamente el conteo de swaps, que domina la sobrecarga de compuertas. El transpilador con IA, en cambio, prioriza la profundidad general y puede ocasionalmente intercambiar algunas compuertas adicionales por una ruta de ejecución más corta.

- **Tiempo de transpilación:** El transpilador con IA tarda más en ejecutarse que el método estándar. Esto se debe al costo computacional adicional de invocar modelos aprendidos durante el enrutamiento y la síntesis. En contraste, el transpilador basado en SABRE es ahora significativamente más rápido después de haber sido reescrito y optimizado en Rust, proporcionando un enrutamiento heurístico altamente eficiente a escala.

Es importante señalar que estos resultados se basan en un solo circuito. Para obtener una comprensión completa de cómo el transpilador con IA se compara con los métodos tradicionales, es necesario probar una variedad de circuitos. El rendimiento de QTS puede variar enormemente dependiendo del tipo de circuito que se está optimizando. Para una comparación más amplia, consulta los benchmarks anteriores o visite el [blog.](https://www.ibm.com/quantum/blog/qiskit-performance)
## Paso 3: Ejecutar utilizando primitivas de Qiskit
Como este tutorial se centra en la transpilación, no se ejecutarán experimentos en el dispositivo cuántico. El objetivo es aprovechar las optimizaciones del Paso 2 para obtener un circuito transpilado con profundidad o conteo de compuertas reducido.
## Paso 4: Post-procesar y devolver el resultado en el formato clásico deseado
Dado que no hay ejecución en este cuaderno, no hay resultados que post-procesar.
# Parte II. Analizar y evaluar los circuitos transpilados
En esta sección, demostraremos cómo analizar el circuito transpilado y evaluarlo comparativamente con la versión original en mayor detalle. Nos centraremos en métricas como la profundidad del circuito, el conteo de compuertas y el tiempo de transpilación para evaluar la efectividad de la optimización. Además, discutiremos cómo los resultados pueden diferir entre varios tipos de circuitos, ofreciendo información sobre el rendimiento más amplio del transpilador en diferentes escenarios.

In [9]:
# -------------------------Step 1-------------------------
num_circuits_hw = 25
depth_hw = 8
qubit_range_hw = list(range(26, 51))

circuits_hw = [
    # We have only two qubit gates, as those test how well the transpiler can optimize the circuit.
    random_circuit(
        num_qubits=n,
        depth=depth_hw,
        max_operands=2,
        num_operand_distribution={2: 1},
        seed=seed + i,
    )
    for i, n in enumerate(qubit_range_hw)
]

print(
    f"Created {len(circuits_hw)} circuits with qubit counts: {qubit_range_hw}"
)

Created 25 circuits with qubit counts: [26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50]


In [10]:
# -------------------------Step 2-------------------------
pm_default = generate_preset_pass_manager(
    optimization_level=3,
    backend=backend,
    seed_transpiler=seed,
)

results_hw = []

for i, qc in enumerate(circuits_hw):
    n = qubit_range_hw[i]

    qc_default, m_default = transpile_with_metrics(pm_default, qc)

    # Create a fresh AI pass manager each iteration to avoid stale layout state
    pm_ai = generate_ai_pass_manager(
        optimization_level=1,
        ai_optimization_level=3,
        backend=backend,
    )
    qc_ai, m_ai = transpile_with_metrics(pm_ai, qc)

    results_hw.append(
        {
            "Qubits": n,
            "Depth 2Q (Default)": m_default["depth_2q"],
            "Depth 2Q (AI)": m_ai["depth_2q"],
            "Gate Count (Default)": m_default["gate_count"],
            "Gate Count (AI)": m_ai["gate_count"],
            "Time (Default)": m_default["time_s"],
            "Time (AI)": m_ai["time_s"],
        }
    )

print_summary(results_hw)

Metric        Default (mean +/- std)     AI (mean +/- std)      AI % improvement
--------------------------------------------------------------------------------
Depth 2Q              217.4 +/- 50.4        191.0 +/- 35.6      +10.9% +/- 10.7%
Gate Count         4513.3 +/- 1394.3     5227.1 +/- 1536.4       -16.4% +/- 5.8%
Time (s)                 0.1 +/- 0.0           3.5 +/- 1.5   -3588.2% +/- 643.6%


In [11]:
plot_metrics_and_pct(results_hw, "Large-Scale Random Circuits")

<Image src="../docs/images/tutorials/ai-transpiler-introduction/extracted-outputs/hw-step2-plot-0.avif" alt="Output of the previous code cell" />

<Image src="../docs/images/tutorials/ai-transpiler-introduction/extracted-outputs/hw-step2-plot-1.avif" alt="Output of the previous code cell" />

In [12]:
# -------------------------Step 3-------------------------
# Build mirror circuits from the 26-qubit case
idx_26q = qubit_range_hw.index(26)

qc_26q = circuits_hw[idx_26q]
qc_default_26q, _ = transpile_with_metrics(pm_default, qc_26q)

pm_ai = generate_ai_pass_manager(
    optimization_level=1,
    ai_optimization_level=3,
    backend=backend,
)
qc_ai_26q, _ = transpile_with_metrics(pm_ai, qc_26q)

mirror_default_hw = build_mirror_circuit(qc_default_26q, simulate=False)
mirror_ai_hw = build_mirror_circuit(qc_ai_26q, simulate=False)

# Re-transpile to basis gates (the inverse can introduce gates like sxdg)
pm_basis = generate_preset_pass_manager(
    optimization_level=0,
    backend=backend,
)
mirror_default_hw = pm_basis.run(mirror_default_hw)
mirror_ai_hw = pm_basis.run(mirror_ai_hw)

print(
    f"Mirror circuit (Default): depth {mirror_default_hw.depth()}, gates {mirror_default_hw.size()}"
)
print(
    f"Mirror circuit (AI):      depth {mirror_ai_hw.depth()}, gates {mirror_ai_hw.size()}"
)

# Submit to real hardware
sampler_hw = SamplerV2(mode=backend)
sampler_hw.options.environment.job_tags = ["TUT_AITI"]

shots_hw = 500000
job_hw = sampler_hw.run([mirror_default_hw, mirror_ai_hw], shots=shots_hw)
print(f"Job submitted: {job_hw.job_id()}")

Mirror circuit (Default): depth 1577, gates 9672
Mirror circuit (AI):      depth 1235, gates 11092
Job submitted: d8gt7vm6983c73dqbg0g


In [13]:
# -------------------------Step 4-------------------------
result_hw = job_hw.result()

survival_probs_hw = {}
for i, method in enumerate(["Default", "AI"]):
    counts = result_hw[i].data.meas.get_counts()
    mirror = [mirror_default_hw, mirror_ai_hw][i]
    all_zeros = "0" * mirror.num_qubits
    survival = counts.get(all_zeros, 0) / shots_hw
    survival_probs_hw[method] = survival
    print(
        f"{method:8s}  P(|00...0>) = {survival:.4f}  ({counts.get(all_zeros, 0)}/{shots_hw})"
    )

# Plot 1 - P(|0...0>), the probability of an erroneous (non-zero) outcome.
# A lower bar means the transpilation introduced less noise.
error_probs_hw = {method: 1 - p for method, p in survival_probs_hw.items()}

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(
    error_probs_hw.keys(),
    error_probs_hw.values(),
    color=["steelblue", "coral"],
)
ax.set_ylabel("1 - P(|0...0>)")
ax.set_title(f"Mirror Circuit Error (26-qubit, {backend.name})")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

Default   P(|00...0>) = 0.0005  (239/500000)
AI        P(|00...0>) = 0.0050  (2516/500000)


<Image src="../docs/images/tutorials/ai-transpiler-introduction/extracted-outputs/hw-step4-1.avif" alt="Output of the previous code cell" />

### Analysis of results

The large-scale results reinforce the trends observed in the small-scale example, now at a more demanding scale.

**Two-qubit depth:** The AI-powered transpiler continues to deliver noticeably lower two-qubit depth across the full range of circuit sizes. Depth optimization is one of the primary objectives the AI routing model is trained on, and the advantage is more pronounced at larger qubit counts where the routing problem becomes harder for heuristic methods.

**Gate count:** The default transpiler (SABRE) consistently produces circuits with fewer gates across all circuit sizes in this range. SABRE's heuristic is specifically designed to minimize gate count, and at this scale the advantage is clear and uniform.

**Transpilation time:** The gap in transpilation time widens at larger scales. SABRE remains nearly constant, while the AI-powered transpiler's runtime grows more steeply. Despite this, the AI-powered transpiler runtime remains practical for most workflows.

**Mirror circuit fidelity:** Both methods produce survival probabilities well under 1% at this scale, leaving little usable signal. With total gate counts around 10,000 and two-qubit depths exceeding 1,000, the depolarizing noise accumulated across the mirror circuit overwhelms most of the signal. This highlights a key limitation of the mirror circuit approach: while it is simple and requires no classical simulation, it does not scale well to large or deep circuits, where both methods are pushed close to the noise floor and the small surviving signal is dominated by accumulated error.

While these results underscore the AI-powered transpiler's effectiveness, it is important to note its limitations. The AI synthesis method is currently only available for certain coupling maps, which may restrict its broader applicability. This constraint should be considered when evaluating its usage in different scenarios.

## Next steps

<Admonition type="tip" title="Recommendations">
If you found this work interesting, you might be interested in the following material:
- [Transpilation optimizations with SABRE](/docs/tutorials/transpilation-optimizations-with-sabre)
- [Compilation methods for Hamiltonian simulation circuits](/docs/tutorials/compilation-methods-for-hamiltonian-simulation-circuits)

</Admonition>